Task 3.1 — Features, split and baseline

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the cleaned dataset
df = pd.read_csv('/content/appointments_clean.csv')

# Select the columns requested for features X
feature_cols = ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel', 'visit_type']
X = df[feature_cols].copy()

# Convert 'reminder_sent' (Yes/No) to 1/0
X['reminder_sent'] = X['reminder_sent'].map({'Yes': 1, 'No': 0})

# One-hot encode booking_channel and visit_type
X = pd.get_dummies(X, columns=['booking_channel', 'visit_type'])

# Set target
y = df['no_show']

# Split into train and test sets (test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Calculate size metrics and baseline accuracy
num_train = len(X_train)
num_test = len(X_test)
no_shows_test = (y_test == 1).sum()

# Baseline of always predicting 'Attended' (i.e. no_show = 0)
baseline_accuracy = (y_test == 0).mean()

print(f"Training set size: {num_train}")
print(f"Test set size: {num_test}")
print(f"Number of no-shows in the test set: {no_shows_test}")
print(f"Baseline accuracy (always predicting 'Attended'): {baseline_accuracy:.4f}")

Training set size: 792
Test set size: 198
Number of no-shows in the test set: 49
Baseline accuracy (always predicting 'Attended'): 0.7525


Task 3.2 — Logistic regression

In [2]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

# 1. Scale features (fit on train only)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 2. Fit Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)

# 3. Predict and calculate metrics
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("=== Performance Metrics ===")
print(f"Accuracy: {accuracy:.4f}")
print("Confusion Matrix:")
print(cm)
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}\n")

# 4. Coefficients
coefficients = model.coef_[0]
feature_names = X.columns

print("=== Model Coefficients ===")
for name, coef in zip(feature_names, coefficients):
    print(f"{name}: {coef:.4f}")
print()

# 5. Odds ratio calculations for reminder_sent
reminder_idx = list(X.columns).index('reminder_sent')
coef_reminder_scaled = coefficients[reminder_idx]
odds_ratio_scaled = np.exp(coef_reminder_scaled)

# Unscaled coefficient: divide by the standard deviation of 'reminder_sent' in the training set
sd_reminder = scaler.scale_[reminder_idx]
coef_reminder_unscaled = coef_reminder_scaled / sd_reminder
odds_ratio_unscaled = np.exp(coef_reminder_unscaled)

print("=== Odds Ratios for reminder_sent ===")
print(f"Scaled Coefficient: {coef_reminder_scaled:.4f}")
print(f"Exp(Scaled Coefficient): {odds_ratio_scaled:.4f}")
print(f"Unscaled Coefficient (divided by SD of {sd_reminder:.4f}): {coef_reminder_unscaled:.4f}")
print(f"Exp(Unscaled Coefficient) [Odds Ratio of No -> Yes]: {odds_ratio_unscaled:.4f}\n")

=== Performance Metrics ===
Accuracy: 0.7879
Confusion Matrix:
[[139  10]
 [ 32  17]]
Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474
ROC-AUC: 0.7889

=== Model Coefficients ===
age: -0.2539
distance_km: 0.2534
fee_inr: 0.4528
lead_days: 0.6145
prior_no_shows: 0.6496
reminder_sent: -0.5807
booking_channel_App: -0.0468
booking_channel_Phone: 0.2159
booking_channel_Walk-In: -0.1864
visit_type_Consultation: 0.0157
visit_type_Follow-up: -0.0157

=== Odds Ratios for reminder_sent ===
Scaled Coefficient: -0.5807
Exp(Scaled Coefficient): 0.5595
Unscaled Coefficient (divided by SD of 0.4950): -1.1731
Exp(Unscaled Coefficient) [Odds Ratio of No -> Yes]: 0.3094



Task 3.3 — Decision tree and overfitting

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1. Unconstrained Decision Tree (no depth limit)
dt_unconstrained = DecisionTreeClassifier(random_state=42)
dt_unconstrained.fit(X_train, y_train)

train_acc_unconstrained = accuracy_score(y_train, dt_unconstrained.predict(X_train))
test_acc_unconstrained = accuracy_score(y_test, dt_unconstrained.predict(X_test))

# 2. Constrained Decision Tree (max_depth=3)
dt_constrained = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_constrained.fit(X_train, y_train)

train_acc_constrained = accuracy_score(y_train, dt_constrained.predict(X_train))
test_acc_constrained = accuracy_score(y_test, dt_constrained.predict(X_test))

print("=== Decision Tree (No Depth Limit) ===")
print(f"Training Accuracy: {train_acc_unconstrained:.4f}")
print(f"Test Accuracy: {test_acc_unconstrained:.4f}")
print(f"Difference: {train_acc_unconstrained - test_acc_unconstrained:.4f}\n")

print("=== Decision Tree (Max Depth = 3) ===")
print(f"Training Accuracy: {train_acc_constrained:.4f}")
print(f"Test Accuracy: {test_acc_constrained:.4f}")
print(f"Difference: {train_acc_constrained - test_acc_constrained:.4f}\n")

=== Decision Tree (No Depth Limit) ===
Training Accuracy: 1.0000
Test Accuracy: 0.6717
Difference: 0.3283

=== Decision Tree (Max Depth = 3) ===
Training Accuracy: 0.8220
Test Accuracy: 0.7778
Difference: 0.0442



Task 3.4 — Random forest and a comparison table

In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

# 1. Fit RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 2. Predict and calculate metrics
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

accuracy_rf = accuracy_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

# 3. Retrieve feature importances
importances = rf.feature_importances_
feature_names = X.columns
rf_importances = pd.Series(importances, index=feature_names).sort_values(ascending=False)

print("=== Random Forest Performance ===")
print(f"Accuracy: {accuracy_rf:.4f}")
print(f"Recall: {recall_rf:.4f}")
print(f"ROC-AUC: {roc_auc_rf:.4f}\n")

print("=== Top 3 Feature Importances ===")
for i, (feat, imp) in enumerate(rf_importances.head(3).items(), 1):
    print(f"{i}. {feat}: {imp:.4f}")
print()

=== Random Forest Performance ===
Accuracy: 0.7475
Recall: 0.3061
ROC-AUC: 0.7057

=== Top 3 Feature Importances ===
1. lead_days: 0.2199
2. age: 0.1968
3. distance_km: 0.1947



In [5]:
# 4. Compile comparative metrics
# Note: Metrics for depth-3 tree are computed here using the previous fit
y_pred_dt3 = dt_constrained.predict(X_test)
y_prob_dt3 = dt_constrained.predict_proba(X_test)[:, 1]

accuracy_dt3 = accuracy_score(y_test, y_pred_dt3)
recall_dt3 = recall_score(y_test, y_pred_dt3)
roc_auc_dt3 = roc_auc_score(y_test, y_prob_dt3)

comparison_data = {
    'Model': ['Logistic Regression', 'Decision Tree (depth-3)', 'Random Forest'],
    'Accuracy': [accuracy, accuracy_dt3, accuracy_rf],
    'Recall': [precision, recall_dt3, recall_rf], # using previous precision as variable index was correct
    'ROC-AUC': [roc_auc, roc_auc_dt3, roc_auc_rf]
}

# Correcting the recall metric for logistic regression which was labeled as recall variable in original scope
comparison_data['Recall'][0] = recall # resetting to the actual recall computed previously

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

print("\n=== Model Selection Conclusion ===")
print("The Logistic Regression model looks best.")

,Model,Accuracy,Recall,ROC-AUC
0,Logistic Regression,0.787879,0.346939,0.788933
1,Decision Tree (depth-3),0.777778,0.224490,0.699699
2,Random Forest,0.747475,0.306122,0.705725



=== Model Selection Conclusion ===
The Logistic Regression model looks best.


Task 3.5 — Tune, choose the model and the threshold, then test once

In [6]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
import pandas as pd

# --- (a) Tune Random Forest ---
param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring="roc_auc")
rf_grid.fit(X_train, y_train)

print("=== (a) Tuned Random Forest ===")
print(f"Best Parameters: {rf_grid.best_params_}")
print(f"Best CV ROC-AUC (best_score_): {rf_grid.best_score_:.4f}\n")

=== (a) Tuned Random Forest ===
Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC (best_score_): 0.7673



In [7]:
# --- (b) Model Selection without touching test set ---
# Logistic Regression Pipeline
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000, random_state=42))])

# 5-fold CV ROC-AUC on training set
cv_auc_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_dt3 = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_rf_default = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_rf_tuned = rf_grid.best_score_

print("=== (b) 5-Fold CV ROC-AUC Scores ===")
print(f"Logistic Regression Pipeline: {cv_auc_lr:.4f}")
print(f"Decision Tree (depth-3):       {cv_auc_dt3:.4f}")
print(f"Random Forest (default):       {cv_auc_rf_default:.4f}")
print(f"Random Forest (tuned):         {cv_auc_rf_tuned:.4f}\n")

winner_name = "Logistic Regression Pipeline"
winner_model = lr_pipe
print(f"Winner Model based on highest cross-validated ROC-AUC: {winner_name}")

=== (b) 5-Fold CV ROC-AUC Scores ===
Logistic Regression Pipeline: 0.7737
Decision Tree (depth-3):       0.6784
Random Forest (default):       0.7466
Random Forest (tuned):         0.7673

Winner Model based on highest cross-validated ROC-AUC: Logistic Regression Pipeline


In [8]:
# --- (c) Choose threshold using out-of-fold training probabilities ---
oof_probs = cross_val_predict(winner_model, X_train, y_train, cv=5, method="predict_proba")[:, 1]

def evaluate_threshold(probs, y_true, thresh):
    preds = (probs >= thresh).astype(int)
    prec = precision_score(y_true, preds)
    rec = recall_score(y_true, preds)
    f1 = f1_score(y_true, preds)
    flagged = preds.sum()
    return prec, rec, f1, flagged

prec_50, rec_50, f1_50, flag_50 = evaluate_threshold(oof_probs, y_train, 0.50)
prec_35, rec_35, f1_35, flag_35 = evaluate_threshold(oof_probs, y_train, 0.35)

print("=== (c) Threshold Tuning (Out-of-Fold) ===")
print(f"Threshold 0.50 | Precision: {prec_50:.4f} | Recall: {rec_50:.4f} | F1: {f1_50:.4f} | Flagged: {flag_50}")
print(f"Threshold 0.35 | Precision: {prec_35:.4f} | Recall: {rec_35:.4f} | F1: {f1_35:.4f} | Flagged: {flag_35}\n")

best_thresh = 0.35 if f1_35 > f1_50 else 0.50
print(f"Selected Threshold: {best_thresh} because it achieves a higher F1 score.")

=== (c) Threshold Tuning (Out-of-Fold) ===
Threshold 0.50 | Precision: 0.5753 | Recall: 0.2485 | F1: 0.3471 | Flagged: 73
Threshold 0.35 | Precision: 0.5130 | Recall: 0.4675 | F1: 0.4892 | Flagged: 154

Selected Threshold: 0.35 because it achieves a higher F1 score.


In [9]:
# --- (d) Evaluate on the test set once ---
# Fit on full training set
winner_model.fit(X_train, y_train)

# Predict on test set
test_probs = winner_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_thresh).astype(int)

test_prec = precision_score(y_test, test_preds)
test_rec = recall_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds)
test_flagged = test_preds.sum()
test_roc_auc = roc_auc_score(y_test, test_probs)

print("=== (d) Final Evaluation on Test Set ===")
print(f"ROC-AUC:       {test_roc_auc:.4f}")
print(f"Precision:     {test_prec:.4f}")
print(f"Recall:        {test_rec:.4f}")
print(f"F1 Score:      {test_f1:.4f}")
print(f"Flagged Count: {test_flagged} out of {len(y_test)}\n")

=== (d) Final Evaluation on Test Set ===
ROC-AUC:       0.7889
Precision:     0.5714
Recall:        0.5714
F1 Score:      0.5714
Flagged Count: 49 out of 198



In [10]:
# --- (e) Save test_risk_scores.csv ---
# Retrieve original appointment_ids for test rows
test_appointment_ids = df.loc[X_test.index, 'appointment_id']

risk_df = pd.DataFrame({
    'appointment_id': test_appointment_ids,
    'no_show': y_test,
    'risk_prob': test_probs,
    'flag': test_preds
})

risk_df.to_csv('test_risk_scores.csv', index=False)
print("=== (e) Saved test_risk_scores.csv ===")
display(risk_df.head())

=== (e) Saved test_risk_scores.csv ===


,appointment_id,no_show,risk_prob,flag
213,A0217,0,0.021565,0
331,A0336,0,0.150917,0
501,A0508,0,0.518692,1
309,A0314,0,0.191300,0
88,A0089,0,0.167718,0
